# Progree AI Internship - Task 3
## Heuristic Graph Pathfinding Agent Search Engine

**What this notebook does**
- Implements **A\* Search** (Manhattan heuristic) and **Dijkstra's algorithm** *from scratch* (only `heapq` is used for the priority queue)
- Tests both agents on 6 different grid-obstacle layouts (open, random 10/20/30%, maze, serpentine corridor)
- Tracks **step count**, **runtime**, **expanded nodes** and **expanded-node density**
- Exports visualization logs: path images, comparison charts, a CSV log, a JSON path log and an animated GIF

Run the cells top to bottom (Runtime → Run all). No uploads needed.

## Step 1: Imports and setup

In [ ]:
import heapq, time, random, json, os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation, PillowWriter

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
SEED = 42
INF = float("inf")
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # 4-connected: up, down, left, right
print("Setup done ✅")

## Step 2: The search agents (from scratch)

Both algorithms share one engine. The only difference is the heuristic:

- **Dijkstra:** priority = `g(n)` (cost so far) → heuristic is always 0, so it explores in all directions
- **A\*:** priority = `g(n) + h(n)` where `h` is the **Manhattan distance** to the goal. This heuristic never overestimates on a 4-connected grid (admissible), so A\* still finds the **optimal** path but expands fewer nodes

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def zero_heuristic(a, b):
    return 0

def search(grid, start, goal, heuristic):
    """Generic best-first search. heuristic=zero_heuristic -> Dijkstra, manhattan -> A*.
    Returns: path (list of cells or None), expansion_order (list of cells in the order expanded)."""
    rows, cols = grid.shape
    counter = 0                                    # tie-breaker so the heap never compares cells
    open_heap = [(heuristic(start, goal), heuristic(start, goal), counter, 0, start)]
    g_score = {start: 0}
    parent = {start: None}
    closed = set()
    order = []

    while open_heap:
        f, h, _, g, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)
        order.append(current)

        if current == goal:                        # rebuild the path
            path, node = [], current
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1], order

        for dr, dc in MOVES:
            nb = (current[0] + dr, current[1] + dc)
            if 0 <= nb[0] < rows and 0 <= nb[1] < cols and grid[nb] == 0 and nb not in closed:
                new_g = g + 1
                if new_g < g_score.get(nb, INF):
                    g_score[nb] = new_g
                    parent[nb] = current
                    counter += 1
                    hn = heuristic(nb, goal)
                    heapq.heappush(open_heap, (new_g + hn, hn, counter, new_g, nb))
    return None, order

def dijkstra(grid, start, goal):
    return search(grid, start, goal, zero_heuristic)

def a_star(grid, start, goal):
    return search(grid, start, goal, manhattan)

# quick sanity test
test = np.zeros((5, 5), dtype=int)
p, o = a_star(test, (0, 0), (4, 4))
print("Sanity check -> path length:", len(p) - 1, "(expected 8) | expanded:", len(o))

## Step 3: Grid obstacle generators

Six different layouts, all 41×41, start at the top-left and goal at the bottom-right. Random grids are re-generated until a path exists.

In [ ]:
def open_grid(n):
    return np.zeros((n, n), dtype=int)

def random_grid(n, density, seed):
    rng = np.random.default_rng(seed)
    while True:
        g = (rng.random((n, n)) < density).astype(int)
        g[0, 0] = 0
        g[n - 1, n - 1] = 0
        if dijkstra(g, (0, 0), (n - 1, n - 1))[0] is not None:
            return g
        seed += 1000
        rng = np.random.default_rng(seed)

def maze_grid(n, seed):
    """Perfect maze via iterative depth-first carving. n must be odd."""
    rnd = random.Random(seed)
    g = np.ones((n, n), dtype=int)
    g[0, 0] = 0
    stack = [(0, 0)]
    while stack:
        r, c = stack[-1]
        nbrs = [(r + dr, c + dc, dr, dc) for dr, dc in [(-2, 0), (2, 0), (0, -2), (0, 2)]
                if 0 <= r + dr < n and 0 <= c + dc < n and g[r + dr, c + dc] == 1]
        if nbrs:
            nr, nc, dr, dc = rnd.choice(nbrs)
            g[r + dr // 2, c + dc // 2] = 0
            g[nr, nc] = 0
            stack.append((nr, nc))
        else:
            stack.pop()
    return g

def serpentine_grid(n):
    """Horizontal walls with alternating gaps force a long winding route."""
    g = np.zeros((n, n), dtype=int)
    for i, r in enumerate(range(2, n - 2, 4)):
        g[r, :] = 1
        if i % 2 == 0:
            g[r, n - 3:] = 0
        else:
            g[r, :3] = 0
    return g

N = 41
START, GOAL = (0, 0), (N - 1, N - 1)
GRIDS = {
    "Open (no obstacles)": open_grid(N),
    "Random 10%": random_grid(N, 0.10, SEED),
    "Random 20%": random_grid(N, 0.20, SEED),
    "Random 30%": random_grid(N, 0.30, SEED),
    "Maze": maze_grid(N, SEED),
    "Serpentine corridor": serpentine_grid(N),
}
for name, g in GRIDS.items():
    print(f"{name:22s} obstacles: {g.mean() * 100:5.1f}%")

## Step 4: Run the experiments and track metrics

For every grid and both algorithms we record:
- **Step count:** number of moves in the final path
- **Runtime:** average over 20 runs (milliseconds)
- **Expanded nodes:** cells taken off the priority queue
- **Expanded-node density:** expanded nodes ÷ free cells × 100 (how much of the map was searched)

In [ ]:
ALGOS = {"Dijkstra": dijkstra, "A* (Manhattan)": a_star}
REPEATS = 20
rows_log, paths_log, search_data = [], {}, {}

for gname, grid in GRIDS.items():
    free_cells = int((grid == 0).sum())
    for aname, fn in ALGOS.items():
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            path, order = fn(grid, START, GOAL)
            times.append((time.perf_counter() - t0) * 1000)
        steps = len(path) - 1
        rows_log.append({
            "grid": gname, "algorithm": aname,
            "path_steps": steps,
            "runtime_ms": round(float(np.mean(times)), 3),
            "expanded_nodes": len(order),
            "free_cells": free_cells,
            "expanded_density_pct": round(100 * len(order) / free_cells, 2),
        })
        paths_log[f"{gname} | {aname}"] = path
        search_data[(gname, aname)] = (path, order)

log_df = pd.DataFrame(rows_log)
log_df.to_csv(f"{OUT_DIR}/search_log.csv", index=False)

# Optimality check: both algorithms must find equally short paths
for gname in GRIDS:
    d = log_df[(log_df.grid == gname) & (log_df.algorithm == "Dijkstra")].path_steps.item()
    a = log_df[(log_df.grid == gname) & (log_df.algorithm == "A* (Manhattan)")].path_steps.item()
    assert d == a, f"Paths differ on {gname}!"
print("Optimality check passed ✅  (A* and Dijkstra found equally short paths on every grid)\n")
log_df

## Step 5: Visualize the paths

Light blue = cells the algorithm expanded, red = final path, green = start, gold = goal, black = obstacles.

In [ ]:
cmap = ListedColormap(["white", "black", "#bfe3f5", "#e63946"])

def draw(ax, grid, order, path, title):
    img = grid.copy()
    for cell in order:
        img[cell] = 2
    for cell in path:
        img[cell] = 3
    ax.imshow(img, cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    ax.scatter(START[1], START[0], c="limegreen", s=60, edgecolors="black", zorder=3)
    ax.scatter(GOAL[1], GOAL[0], c="gold", s=60, edgecolors="black", zorder=3)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

for gname, grid in GRIDS.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, aname in zip(axes, ALGOS):
        path, order = search_data[(gname, aname)]
        draw(ax, grid, order, path, f"{aname}\nsteps={len(path)-1}, expanded={len(order)}")
    fig.suptitle(gname, fontweight="bold")
    plt.tight_layout()
    fname = gname.replace(" ", "_").replace("%", "pct").replace("(", "").replace(")", "")
    plt.savefig(f"{OUT_DIR}/path_{fname}.png", dpi=150)
    plt.show()

## Step 6: Comparison charts (expanded nodes, runtime, density)

In [ ]:
metrics = [("expanded_nodes", "Expanded nodes"),
           ("runtime_ms", "Runtime (ms)"),
           ("expanded_density_pct", "Expanded-node density (%)")]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
x = np.arange(len(GRIDS)); w = 0.38
for ax, (col, label) in zip(axes, metrics):
    for i, aname in enumerate(ALGOS):
        vals = [log_df[(log_df.grid == g) & (log_df.algorithm == aname)][col].item() for g in GRIDS]
        ax.bar(x + (i - 0.5) * w, vals, w, label=aname)
    ax.set_xticks(x)
    ax.set_xticklabels(list(GRIDS), rotation=35, ha="right", fontsize=8)
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/metrics_comparison.png", dpi=150)
plt.show()

# Summary: how much work A* saves
piv = log_df.pivot(index="grid", columns="algorithm", values="expanded_nodes")
piv["A* saves (%)"] = (100 * (1 - piv["A* (Manhattan)"] / piv["Dijkstra"])).round(1)
piv

## Step 7: Scalability test

How do the algorithms behave as the grid grows? (Random grids, 20% obstacles)

In [ ]:
sizes = [20, 40, 80, 120, 160]
scale_rows = []
for n in sizes:
    g = random_grid(n, 0.20, SEED)
    s, t = (0, 0), (n - 1, n - 1)
    for aname, fn in ALGOS.items():
        runs = []
        for _ in range(5):
            t0 = time.perf_counter()
            path, order = fn(g, s, t)
            runs.append((time.perf_counter() - t0) * 1000)
        ms = float(np.mean(runs))
        scale_rows.append({"size": f"{n}x{n}", "n": n, "algorithm": aname,
                           "path_steps": len(path) - 1, "expanded_nodes": len(order),
                           "runtime_ms": round(ms, 2)})
scale_df = pd.DataFrame(scale_rows)
scale_df.drop(columns="n").to_csv(f"{OUT_DIR}/scalability_log.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for aname in ALGOS:
    d = scale_df[scale_df.algorithm == aname]
    axes[0].plot(d.n, d.expanded_nodes, marker="o", label=aname)
    axes[1].plot(d.n, d.runtime_ms, marker="o", label=aname)
axes[0].set_title("Expanded nodes vs grid size"); axes[1].set_title("Runtime (ms) vs grid size")
for ax in axes:
    ax.set_xlabel("Grid side length"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/scalability.png", dpi=150)
plt.show()
scale_df.drop(columns="n")

## Step 8: Animated visualization log (path optimization over time)

A GIF showing how both searches grow on the *Random 20%* grid, side by side.

In [ ]:
gname = "Random 20%"
grid = GRIDS[gname]
d_path, d_order = search_data[(gname, "Dijkstra")]
a_path, a_order = search_data[(gname, "A* (Manhattan)")]
FRAMES = 40
total = max(len(d_order), len(a_order))
step = max(1, total // FRAMES)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.8))
def render(frame):
    k = frame * step
    for ax, order, path, name in [(axes[0], d_order, d_path, "Dijkstra"),
                                  (axes[1], a_order, a_path, "A* (Manhattan)")]:
        ax.clear()
        shown = order[:k]
        finished = k >= len(order)
        draw(ax, grid, shown, path if finished else [], f"{name}\nexpanded: {min(k, len(order))}")
    return axes

anim = FuncAnimation(fig, render, frames=FRAMES + 3, interval=150)
anim.save(f"{OUT_DIR}/search_animation.gif", writer=PillowWriter(fps=7))
plt.close(fig)

from IPython.display import Image, display
display(Image(filename=f"{OUT_DIR}/search_animation.gif"))

## Step 9: Export the path log and download everything

In [ ]:
with open(f"{OUT_DIR}/paths_log.json", "w") as f:
    json.dump({k: [list(c) for c in v] for k, v in paths_log.items()}, f)

with open(f"{OUT_DIR}/summary.txt", "w") as f:
    f.write("PATHFINDING EXPERIMENT SUMMARY\n" + "=" * 32 + "\n\n")
    f.write(log_df.to_string(index=False))
    f.write("\n\nA* expansion savings vs Dijkstra (%):\n")
    f.write(piv["A* saves (%)"].to_string())

print("Files in results/:")
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn)

with zipfile.ZipFile("task3_results.zip", "w") as z:
    for fn in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, fn))
try:
    from google.colab import files
    files.download("task3_results.zip")
except ImportError:
    pass

## Observations (write these in your README / report)

- **Optimality:** both algorithms return paths of identical length on every grid, which confirms the Manhattan heuristic is admissible.
- **Efficiency:** A\* expands fewer nodes than Dijkstra on open and lightly obstructed grids because the heuristic steers the search toward the goal.
- **Where the gap shrinks:** on mazes and serpentine corridors there is only one route, so both algorithms have to explore almost the same cells.
- **Scaling:** as the grid grows, Dijkstra's expanded nodes and runtime grow faster than A\*'s.

*(Check these against your own tables and charts before submitting, and edit if your numbers show something different.)*